# Ground-Truth Driver / Non-Driver Gene Retrieval

Retrieves the ground-truth driver and non-driver gene sets (protein-coding universe -> driver
exclusion -> mutation-frequency filter -> disease-pathway filter -> length/expression-matched
negative sampling) and saves **three separate files**:

1. `driver_genes.txt` -- one gene per line, the ground-truth positive (driver) set
2. `non_driver_genes.txt` -- one gene per line, the ground-truth negative (non-driver) set
3. `gene_labels.csv` -- both combined into one labeled table (`label` column: 1 = driver, 0 = non-driver)

This uses the full pipeline including **matched negative sampling**
(`config.USE_MATCHED_NEGATIVE_SAMPLING`), so non-driver genes are matched to the positives'
length/expression (and optionally mutation-rate) distribution rather than left as "whatever
survived the filters." For the unmatched comparison set instead, see
`driver_labeling_unmatched_baseline.ipynb`.


In [ ]:
import os
import random
import sys
from pathlib import Path

import pandas as pd

# Notebooks live in notebooks/, but config.py's paths (e.g. "data/...") are
# relative to the repo root -- chdir there so those paths resolve correctly
# regardless of where Jupyter's working directory starts out.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import config
from src.driver_labeling import (
    apply_mutation_frequency_filter,
    apply_pathway_filter,
    build_gene_labels,
    load_driver_genes,
)
from src.gene_universe import build_protein_coding_gene_universe
from src.negative_sampling import (
    build_covariate_table,
    build_driver_exclusion_set,
    compute_gene_lengths,
    load_gene_expression,
    load_gene_mutation_rate,
    match_negatives,
    summarize_matching_quality,
)
from src.pathway_filter import build_disease_pathway_genes

random.seed(config.RANDOM_SEED)
config.DATA_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] USE_MUTATION_FILTER: {config.USE_MUTATION_FILTER}")
print(f"[CONFIG] USE_PATHWAY_FILTER: {config.USE_PATHWAY_FILTER}")
print(f"[CONFIG] USE_MATCHED_NEGATIVE_SAMPLING: {config.USE_MATCHED_NEGATIVE_SAMPLING}")
print(f"[CONFIG] MATCH_ON_MUTATION_RATE: {config.MATCH_ON_MUTATION_RATE}")


## 1. Protein-coding gene universe

Parses the GENCODE GTF and keeps only `gene_type == "protein_coding"`.


In [ ]:
all_genes = build_protein_coding_gene_universe(
    "../../data/gencode.v49.basic.annotation.gtf"
)
pd.DataFrame(sorted(all_genes), columns=["gene_name"]).to_csv(
    config.GENCODE_GENES_FILE, index=False
)
print(f"[GENE UNIVERSE] {len(all_genes)} protein-coding genes")


## 2. Ground-truth driver genes and the expanded exclusion set

Positives (ground-truth drivers) come from the COSMIC Cancer Gene Census. The exclusion set
used to derive raw non-driver candidates is broader -- NCG + CGC + IntOGen + Bailey et al. 2018
-- so a gene isn't kept as a "negative" just because it's missing from CGC specifically.


In [ ]:
driver_genes = load_driver_genes("../../data/Census_allWed.tsv")
print(f"[DRIVERS] {len(driver_genes)} ground-truth CGC driver genes")

driver_exclusion_set = build_driver_exclusion_set(
    ncg_file=config.NCG_FILE if config.NCG_FILE.exists() else None,
    cgc_file=config.CGC_CENSUS_FILE if config.CGC_CENSUS_FILE.exists() else None,
    intogen_file=config.INTOGEN_FILE if config.INTOGEN_FILE.exists() else None,
    bailey_file=config.BAILEY_FILE if config.BAILEY_FILE.exists() else None,
)

non_drivers = all_genes - driver_exclusion_set
print(
    f"[NON-DRIVERS] {len(non_drivers)} raw candidates "
    "(excluded from NCG/CGC/IntOGen/Bailey)"
)


## 3. Mutation-frequency filter

Excludes any candidate with mutation frequency >= `config.MUTATION_FREQUENCY_THRESHOLD` in any
TCGA cancer type. Skipped entirely if `config.USE_MUTATION_FILTER` is `False`.


In [ ]:
if config.USE_MUTATION_FILTER:
    before = len(non_drivers)
    non_drivers = apply_mutation_frequency_filter(
        non_drivers,
        config.MUTATION_FREQUENCY_FILE,
        threshold=config.MUTATION_FREQUENCY_THRESHOLD,
    )
    print(
        f"[MUTATION FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )
else:
    print("[MUTATION FILTER] skipped (config.USE_MUTATION_FILTER is False)")


## 4. Disease-pathway filter

Excludes candidates belonging to a Reactome pathway that is a descendant of the top-level
*Disease* pathway. Skipped entirely if `config.USE_PATHWAY_FILTER` is `False`.


In [ ]:
if config.USE_PATHWAY_FILTER:
    pathway_genes = build_disease_pathway_genes(
        "../../data/reactome/ReactomePathways.gmt",
        "../../data/reactome/reactome_relations.csv",
    )
    pd.DataFrame(sorted(pathway_genes), columns=["gene"]).to_csv(
        config.DISEASE_PATHWAY_GENES_FILE, index=False
    )
    print(f"[PATHWAY FILTER] {len(pathway_genes)} disease-pathway genes")

    before = len(non_drivers)
    non_drivers = apply_pathway_filter(non_drivers, pathway_genes)
    print(
        f"[PATHWAY FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )
else:
    print("[PATHWAY FILTER] skipped (config.USE_PATHWAY_FILTER is False)")


## 5. Length- and expression-matched negative sampling

Rather than keeping every surviving candidate, sample negatives matched to the positives'
length/expression (and optionally mutation-rate) covariate distribution. Skipped entirely if
`config.USE_MATCHED_NEGATIVE_SAMPLING` is `False` -- in that case `non_drivers` stays as
whatever survived the filters above.


In [ ]:
if config.USE_MATCHED_NEGATIVE_SAMPLING:
    gene_lengths = compute_gene_lengths("../../data/gencode.v49.basic.annotation.gtf")
    gene_expression = load_gene_expression("../../data/gene_median_expression.csv")
    mutation_rate = (
        load_gene_mutation_rate(config.MUTATION_FREQUENCY_FILE)
        if config.MATCH_ON_MUTATION_RATE
        else None
    )

    match_df = match_negatives(
        positive_genes=driver_genes,
        candidate_negative_genes=non_drivers,
        gene_lengths=gene_lengths,
        gene_expression=gene_expression,
        mutation_rate=mutation_rate,
        n_per_positive=config.NEGATIVES_PER_POSITIVE,
        random_seed=config.RANDOM_SEED,
    )
    matched_negatives = set(match_df["matched_negative_gene"])
    print(
        f"[MATCHED NEGATIVES] {len(matched_negatives)} negatives matched to "
        f"{len(driver_genes)} positives"
    )
    match_df.to_csv(config.PROCESSED_DIR / "negative_gene_matches.csv", index=False)

    pos_covariates = build_covariate_table(
        driver_genes, gene_lengths, gene_expression, mutation_rate
    )
    quality_report = summarize_matching_quality(
        pos_df=pos_covariates,
        matched_negative_genes=matched_negatives,
        unmatched_candidate_genes=non_drivers,
        gene_lengths=gene_lengths,
        gene_expression=gene_expression,
        mutation_rate=mutation_rate,
    )
    quality_report.to_csv(config.PROCESSED_DIR / "negative_matching_quality.csv", index=False)

    non_drivers = matched_negatives
else:
    print("[MATCHED SAMPLING] skipped (config.USE_MATCHED_NEGATIVE_SAMPLING is False)")

print(f"[FINAL NON-DRIVERS] {len(non_drivers)} ground-truth negative genes")


## 6. Save three separate files

`driver_genes.txt` and `non_driver_genes.txt` hold each gene set on its own (one gene per
line); `gene_labels.csv` combines both into a single labeled table for anything downstream
that wants the pair together.


In [ ]:
driver_path = config.PROCESSED_DIR / "driver_genes.txt"
nondriver_path = config.PROCESSED_DIR / "non_driver_genes.txt"

driver_path.write_text("\n".join(sorted(driver_genes)) + "\n")
nondriver_path.write_text("\n".join(sorted(non_drivers)) + "\n")

labels_df = build_gene_labels(driver_genes, non_drivers)
print(labels_df["label"].value_counts())
labels_df.to_csv(config.GENE_LABELS_FILE, index=False)

print(f"[DONE] Drivers ({len(driver_genes)}): {driver_path}")
print(f"[DONE] Non-drivers ({len(non_drivers)}): {nondriver_path}")
print(f"[DONE] Combined labels: {config.GENE_LABELS_FILE}")

labels_df.head()
